# 6장 Prompt Engineering — SageMaker AI Studio(JupyterLab)용

<!-- origin-notice -->
> **원본 노트북:** [chapter06/Chapter 6 - Prompt Engineering.ipynb](https://github.com/HandsOnLLM/Hands-On-Large-Language-Models/blob/main/chapter06/Chapter%206%20-%20Prompt%20Engineering.ipynb)  
> **원본 저장소:** [HandsOnLLM/Hands-On-Large-Language-Models](https://github.com/HandsOnLLM/Hands-On-Large-Language-Models) (Apache License 2.0, © 2024 Jay Alammar & Maarten Grootendorst)  
> 이 노트북은 원본을 SageMaker AI Studio에서 실행되도록 수정하고 한국어 설명·보충 실험을 추가한 것입니다. 변경 내용은 노트북 안의 "책과 다를 수 있는 점"에 적혀 있습니다.

원본: HandsOnLLM/Hands-On-Large-Language-Models `chapter06` 노트북(Apache 2.0, 이 노트북은 그 내용을 수정·보충한 것). 권장 환경은 `ml.g4dn.xlarge`(T4 16GB)이고, **스토리지는 40GB 이상**을 권합니다. 6절에서 Phi-3의 GGUF 파일(fp16 7.6GB)을 하나 더 내려받기 때문입니다. 스토리지가 30GB라면 6절에서 q4 파일(2.4GB)을 고르세요.

| 절 | 하는 일 | 책 위치(PDF 쪽) |
|---|---|---|
| 1 | Phi-3 불러오기, 채팅 템플릿 확인 | 236–240 |
| 2 | 출력 제어: temperature, top_p | 240–244 |
| 3 | 프롬프트의 기본 재료와 지시 기반 프롬프팅 | 244–249 |
| 4 | 복잡한 프롬프트, 인컨텍스트 학습(예시 제공), 체인 프롬프팅 | 250–259 |
| 5 | 추론: Chain-of-Thought, Zero-shot CoT, Tree-of-Thought (+ Self-Consistency 구현) | 260–269 |
| 6 | 출력 검증: 예시로 형식 유도, 문법 제약 샘플링(llama.cpp) (+ JSON 검증·재시도, 스키마 제약) | 270–277 |

`(+ …)` 표시는 책에 없는 보충 실험입니다. 셀은 위에서부터 차례로 실행하세요.

**용어.** 프롬프트 엔지니어링(prompt engineering)은 모델 가중치를 바꾸지 않고 **입력 문장만 설계해서** 원하는 출력을 얻는 기법입니다. 모델을 다시 학습하는 미세조정(fine-tuning, 12장)과 대비됩니다.

## 0. 환경 점검

GPU·디스크·버전을 확인합니다. 채팅 템플릿(`apply_chat_template`)은 jinja2 3.1 이상이 필요합니다. llama.cpp는 6절에서 필요할 때 설치합니다.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h ~ | tail -1
import importlib, subprocess, sys
import jinja2
if tuple(map(int, jinja2.__version__.split(".")[:2])) < (3, 1):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "jinja2>=3.1"])
import torch, transformers
print("torch", torch.__version__, "| CUDA", torch.cuda.is_available(), "| transformers", transformers.__version__)

## 1. 텍스트 생성 모델 불러오기 (PDF 236–240)

책은 모델을 고를 때 **작은 모델로 시작해 필요할 때 키우라**고 권합니다(236쪽). 이 장도 1장과 같은 Phi-3-mini(38억 파라미터)를 씁니다. `do_sample=False`는 매번 가장 확률이 높은 토큰을 고르는 탐욕(greedy) 디코딩입니다.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    device_map="cuda",
    torch_dtype="auto",        # transformers 5.x에서는 dtype="auto" (기존 이름은 경고만 나고 동작)
    trust_remote_code=False,   # 책 본문은 True. 최신 transformers에는 Phi-3가 내장되어 False가 안전
)
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,
    max_new_tokens=500,
    do_sample=False,
)

In [ ]:
messages = [{"role": "user", "content": "Create a funny joke about chickens."}]
output = pipe(messages)
print(output[0]["generated_text"])

`messages`가 실제로 어떤 문자열이 되어 모델에 들어가는지 봅니다. `<|user|>`, `<|end|>`, `<|assistant|>`는 Phi-3가 학습 때 본 특수 토큰으로, 누가 말하는지와 말이 끝나는 지점을 표시합니다(2장 1절).

> **책과 다를 수 있는 점.** 책(239쪽)의 출력은 맨 앞에 `<s>`가 붙어 있습니다. 현재 Hugging Face의 Phi-3 토크나이저 설정에서는 붙지 않습니다(2장 노트북에서 확인한 것과 같은 차이).

In [ ]:
prompt = pipe.tokenizer.apply_chat_template(messages, tokenize=False)
print(prompt)

## 2. 출력 제어: temperature와 top_p (PDF 240–244)

샘플링을 켜면(`do_sample=True`) 모델이 매 단계 확률 분포에서 토큰을 **뽑습니다**.

- **temperature:** 분포를 평평하게(높게) 또는 뾰족하게(낮게) 만듭니다. 높을수록 확률 낮은 토큰도 자주 뽑혀 다양하고 엉뚱해집니다.
- **top_p(nucleus sampling):** 확률 높은 순으로 더해 누적합이 p가 될 때까지의 토큰만 후보로 둡니다. 0.1이면 소수 후보만, 1이면 전체가 후보입니다. 비슷한 `top_k`는 후보 수를 정확히 k개로 자릅니다.

관찰 포인트: 아래 두 셀을 여러 번 실행해 보세요. 실행할 때마다 다른 농담이 나옵니다. `do_sample=False`로 둔 1절 셀은 몇 번을 실행해도 같습니다.

In [ ]:
output = pipe(messages, do_sample=True, temperature=1)
print(output[0]["generated_text"])

In [ ]:
output = pipe(messages, do_sample=True, top_p=1)
print(output[0]["generated_text"])

책의 표 6-1(244쪽)은 용도별 조합을 이렇게 정리합니다.

| 용도 | temperature | top_p | 성격 |
|---|---|---|---|
| 브레인스토밍 | 높게 | 높게 | 가장 다양하고 예상 밖의 결과 |
| 이메일 작성 | 낮게 | 낮게 | 예측 가능하고 보수적 |
| 창작 | 높게 | 낮게 | 창의적이되 후보가 좁아 일관성 유지 |
| 번역 | 낮게 | 높게 | 결정적이되 어휘 폭은 넓게 |

## 3. 프롬프트의 기본 재료 (PDF 244–249)

이 절은 코드 없이 개념만 다룹니다. 책이 말하는 기본 재료는 **지시(instruction)**와 **데이터**이고, 출력 형식을 정해 주는 **출력 지시자(output indicator)**를 더하면 결과가 안정됩니다. 지시 기반 프롬프팅에서 책이 강조하는 세 가지는 다음과 같습니다.

- **구체성(specificity):** "요약해"보다 "두 문장으로, 결론부터" 처럼 정확히 무엇을 원하는지 쓴다.
- **환각(hallucination) 억제:** "모르면 모른다고 답하라"를 명시한다.
- **순서:** 모델은 프롬프트의 처음(primacy effect)과 끝(recency effect)에 있는 정보에 더 주의를 둔다. 긴 프롬프트에서는 지시를 맨 앞이나 맨 끝에 둔다.

## 4. 고급 프롬프트 엔지니어링 (PDF 250–259)

### 4-1. 복잡한 프롬프트: 재료를 모듈로 조립하기 (PDF 251–254)

페르소나(persona), 지시, 맥락(context), 형식, 청중(audience), 어조(tone), 데이터를 각각 문자열로 만들고 이어 붙입니다. 모듈마다 넣고 빼며 출력이 어떻게 바뀌는지 실험하기 쉽게 하려는 구성입니다.

> **책과 다를 수 있는 점.** 책(254쪽)과 공식 노트북은 요약할 긴 글을 `text`에 넣은 직후 `text = "MY TEXT TO SUMMARIZE"`로 다시 덮어씁니다. 그대로 실행하면 모델은 실제 글 대신 이 자리표시 문장을 요약하게 됩니다. 공식 노트북의 저장된 출력은 Transformer 요약이라, 덮어쓰기 줄 없이 실행한 결과로 보입니다. 여기서는 그 줄을 주석으로 막았습니다.

In [ ]:
# 요약할 글 (출처: https://jalammar.github.io/illustrated-transformer/ 의 앞부분)
text = """In the previous post, we looked at Attention – a ubiquitous method in modern deep learning models. Attention is a concept that helped improve the performance of neural machine translation applications. In this post, we will look at The Transformer – a model that uses attention to boost the speed with which these models can be trained. The Transformer outperforms the Google Neural Machine Translation model in specific tasks. The biggest benefit, however, comes from how The Transformer lends itself to parallelization. It is in fact Google Cloud’s recommendation to use The Transformer as a reference model to use their Cloud TPU offering. So let’s try to break the model apart and look at how it functions.
The Transformer was proposed in the paper Attention is All You Need. A TensorFlow implementation of it is available as a part of the Tensor2Tensor package. Harvard’s NLP group created a guide annotating the paper with PyTorch implementation. In this post, we will attempt to oversimplify things a bit and introduce the concepts one by one to hopefully make it easier to understand to people without in-depth knowledge of the subject matter.
Let’s begin by looking at the model as a single black box. In a machine translation application, it would take a sentence in one language, and output its translation in another.
Popping open that Optimus Prime goodness, we see an encoding component, a decoding component, and connections between them.
The encoding component is a stack of encoders (the paper stacks six of them on top of each other – there’s nothing magical about the number six, one can definitely experiment with other arrangements). The decoding component is a stack of decoders of the same number.
The encoders are all identical in structure (yet they do not share weights). Each one is broken down into two sub-layers:
The encoder’s inputs first flow through a self-attention layer – a layer that helps the encoder look at other words in the input sentence as it encodes a specific word. We’ll look closer at self-attention later in the post.
The outputs of the self-attention layer are fed to a feed-forward neural network. The exact same feed-forward network is independently applied to each position.
The decoder has both those layers, but between them is an attention layer that helps the decoder focus on relevant parts of the input sentence (similar what attention does in seq2seq models).
Now that we’ve seen the major components of the model, let’s start to look at the various vectors/tensors and how they flow between these components to turn the input of a trained model into an output.
As is the case in NLP applications in general, we begin by turning each input word into a vector using an embedding algorithm.
Each word is embedded into a vector of size 512. We'll represent those vectors with these simple boxes.
The embedding only happens in the bottom-most encoder. The abstraction that is common to all the encoders is that they receive a list of vectors each of the size 512 – In the bottom encoder that would be the word embeddings, but in other encoders, it would be the output of the encoder that’s directly below. The size of this list is hyperparameter we can set – basically it would be the length of the longest sentence in our training dataset.
After embedding the words in our input sequence, each of them flows through each of the two layers of the encoder.
Here we begin to see one key property of the Transformer, which is that the word in each position flows through its own path in the encoder. There are dependencies between these paths in the self-attention layer. The feed-forward layer does not have those dependencies, however, and thus the various paths can be executed in parallel while flowing through the feed-forward layer.
Next, we’ll switch up the example to a shorter sentence and we’ll look at what happens in each sub-layer of the encoder.
Now We’re Encoding!
As we’ve mentioned already, an encoder receives a list of vectors as input. It processes this list by passing these vectors into a ‘self-attention’ layer, then into a feed-forward neural network, then sends out the output upwards to the next encoder.
"""

# 프롬프트 재료
persona = "You are an expert in Large Language models. You excel at breaking down complex papers into digestible summaries.\n"
instruction = "Summarize the key findings of the paper provided.\n"
context = "Your summary should extract the most crucial points that can help researchers quickly understand the most vital information of the paper.\n"
data_format = "Create a bullet-point summary that outlines the method. Follow this up with a concise paragraph that encapsulates the main results.\n"
audience = "The summary is designed for busy researchers that quickly need to grasp the newest trends in Large Language Models.\n"
tone = "The tone should be professional and clear.\n"
# text = "MY TEXT TO SUMMARIZE"   # 책·공식 노트북에 있는 줄. 위 글을 덮어쓰므로 막아 둠
data = f"Text to summarize: {text}"

# 재료를 빼거나 순서를 바꿔 가며 출력 변화를 보세요
query = persona + instruction + context + data_format + audience + tone + data

In [ ]:
messages = [{"role": "user", "content": query}]
print(tokenizer.apply_chat_template(messages, tokenize=False)[:600], "...")

In [ ]:
outputs = pipe(messages)
print(outputs[0]["generated_text"])

관찰 포인트: 형식 지시대로 "글머리표 요약 + 결과 문단" 구조가 나오는지, `max_new_tokens=500`에 걸려 끝이 잘리지는 않는지 보세요. 공식 노트북의 저장된 출력도 마지막 문단이 중간에 끊겨 있습니다.

### 4-2. 인컨텍스트 학습: 예시 제공 (PDF 255–257)

모델에게 예시를 보여 주면 가중치를 바꾸지 않고도 그 형식을 따라 합니다. 예시 0개는 zero-shot, 1개는 one-shot, 여러 개는 few-shot입니다. 여기서는 지어낸 단어("Gigamuru")로 문장을 만드는 예시 하나를 user/assistant 대화로 넣고, 새 지어낸 단어("screeg")에 대해 같은 일을 시킵니다.

In [ ]:
one_shot_prompt = [
    {"role": "user", "content": "A 'Gigamuru' is a type of Japanese musical instrument. An example of a sentence that uses the word Gigamuru is:"},
    {"role": "assistant", "content": "I have a Gigamuru that my uncle gave me as a gift. I love to play it at home."},
    {"role": "user", "content": "To 'screeg' something is to swing a sword at it. An example of a sentence that uses the word screeg is:"},
]
print(tokenizer.apply_chat_template(one_shot_prompt, tokenize=False))

In [ ]:
outputs = pipe(one_shot_prompt)
print(outputs[0]["generated_text"])

### 4-3. 체인 프롬프팅: 문제 나누기 (PDF 257–259)

한 번에 다 시키지 않고, 앞 단계 출력을 다음 프롬프트의 입력으로 넘깁니다. 여기서는 (1) 제품 이름과 슬로건 → (2) 그걸 바탕으로 판매 문구. 단계마다 프롬프트와 파라미터를 따로 조절할 수 있다는 점이 장점입니다. 7장의 LangChain Chain이 이 패턴을 코드 구조로 만든 것입니다.

In [ ]:
product_prompt = [{"role": "user", "content": "Create a name and slogan for a chatbot that leverages LLMs."}]
outputs = pipe(product_prompt)
product_description = outputs[0]["generated_text"]
print(product_description)

In [ ]:
sales_prompt = [{"role": "user", "content": f"Generate a very short sales pitch for the following product: '{product_description}'"}]
outputs = pipe(sales_prompt)
sales_pitch = outputs[0]["generated_text"]
print(sales_pitch)

## 5. 생성 모델로 추론하기 (PDF 260–269)

여기서 추론은 reasoning(단계를 밟아 답에 이르는 사고)을 뜻하며, 모델을 실행하는 inference와 다릅니다.

### 5-1. 비교 기준: 추론 없이 바로 답하게 하기 (PDF 261)

예시 문제에 답만 보여 주고("11") 새 문제를 냅니다. 책은 이 비교를 그림으로만 보여 주고 코드는 싣지 않았고, 공식 노트북에는 주석으로 막힌 셀로 남아 있습니다. 5-2와 비교하려고 살렸습니다.

In [ ]:
standard_prompt = [
    {"role": "user", "content": "Roger has 5 tennis balls. He buys 2 more cans of tennis balls. Each can has 3 tennis balls. How many tennis balls does he have now?"},
    {"role": "assistant", "content": "11"},
    {"role": "user", "content": "The cafeteria had 23 apples. If they used 20 to make lunch and bought 6 more, how many apples do they have?"},
]
outputs = pipe(standard_prompt)
print(outputs[0]["generated_text"])

> **책과 다를 수 있는 점.** 공식 노트북에 주석으로 남은 이 셀은 문제의 사과 개수가 **25개**로 되어 있어 책(23개)과 다릅니다. 여기서는 책과 같은 23개로 맞췄습니다. 정답은 23 − 20 + 6 = **9**입니다.

### 5-2. Chain-of-Thought: 답하기 전에 생각하기 (PDF 261–264)

예시의 답에 **풀이 과정**까지 보여 줍니다. 모델은 형식을 따라 풀이부터 쓰게 되고, 풀이에서 생성한 토큰이 다음 토큰의 근거가 됩니다. 1장에서 본 자기회귀 생성 때문에, 중간 계산을 글로 쓰는 것 자체가 계산을 돕는 셈입니다.

In [ ]:
cot_prompt = [
    {"role": "user", "content": "Roger has 5 tennis balls. He buys 2 more cans of tennis balls. Each can has 3 tennis balls. How many tennis balls does he have now?"},
    {"role": "assistant", "content": "Roger started with 5 balls. 2 cans of 3 tennis balls each is 6 tennis balls. 5 + 6 = 11. The answer is 11."},
    {"role": "user", "content": "The cafeteria had 23 apples. If they used 20 to make lunch and bought 6 more, how many apples do they have?"},
]
outputs = pipe(cot_prompt)
print(outputs[0]["generated_text"])

### 5-3. Zero-shot Chain-of-Thought (PDF 264–265)

예시 없이 "Let's think step-by-step."만 붙여도 단계별 풀이를 끌어낼 수 있습니다.

In [ ]:
zeroshot_cot_prompt = [
    {"role": "user", "content": "The cafeteria had 23 apples. If they used 20 to make lunch and bought 6 more, how many apples do they have? Let's think step-by-step."}
]
outputs = pipe(zeroshot_cot_prompt)
print(outputs[0]["generated_text"])

### 5-4. (+ 보충) Self-Consistency: 여러 번 뽑아 다수결 (PDF 265–266)

책은 개념만 설명하고 코드를 싣지 않습니다. 같은 CoT 프롬프트를 **샘플링을 켜고** 여러 번 실행해, 각 풀이의 최종 숫자 중 가장 많이 나온 값을 답으로 고릅니다. 풀이 하나가 틀려도 다수가 맞으면 답을 바로잡을 수 있지만, 실행 횟수만큼 비용과 시간이 늘어납니다. 또 다수결은 풀이들이 **서로 독립적으로** 맞을 가능성이 높을 때만 효과가 있습니다. 모델이 같은 방식으로 틀리면 다수결도 틀립니다. 제 작업 환경에서 1억 3천만 파라미터 모델로 시험했을 때는 5번 중 4번이 같은 오답(26)을 내 다수결 결과도 26이었습니다.

In [ ]:
import re
from collections import Counter

def final_number(s):
    m = re.search(r"answer is\D*(-?\d+)", s, flags=re.I)
    nums = re.findall(r"-?\d+", s)
    return int(m.group(1)) if m else (int(nums[-1]) if nums else None)

N = 5
answers = []
for i in range(N):
    out = pipe(cot_prompt, do_sample=True, temperature=0.8, top_p=0.95, max_new_tokens=150)[0]["generated_text"]
    answers.append(final_number(out))
    print(f"[{i}] {answers[-1]!s:>5} | {out.strip()[:120]!r}")
vote = Counter(a for a in answers if a is not None).most_common()
print("투표 결과:", vote, "→ 최종 답:", vote[0][0] if vote else None)

### 5-5. Tree-of-Thought: 중간 단계 탐색 (PDF 266–269)

원래 Tree-of-Thought는 여러 풀이 갈래를 만들고 평가하며 가지치기를 반복하므로 모델을 여러 번 부릅니다. 책은 이를 **프롬프트 하나로 흉내 내는** 방법을 소개합니다. "세 명의 전문가가 한 단계씩 생각을 공유하고, 틀린 사람은 빠진다"는 설정입니다.

In [ ]:
zeroshot_tot_prompt = [
    {"role": "user", "content": "Imagine three different experts are answering this question. All experts will write down 1 step of their thinking, then share it with the group. Then all experts will go on to the next step, etc. If any expert realises they're wrong at any point then they leave. The question is 'The cafeteria had 23 apples. If they used 20 to make lunch and bought 6 more, how many apples do they have?' Make sure to discuss the results."}
]
outputs = pipe(zeroshot_tot_prompt)
print(outputs[0]["generated_text"])

## 6. 출력 검증 (PDF 270–277)

실서비스에서는 출력이 정해진 **구조**(예: JSON)를 따르는지, 허용된 값만 쓰는지, 안전한지, 정확한지를 확인해야 합니다. 책은 출력을 통제하는 방법으로 예시 제공, 문법(grammar) 제약, 미세조정(12장)의 세 가지를 들고, 이 장에서는 앞의 두 가지를 다룹니다.

### 6-1. 예시로 형식 유도하기 (PDF 272–274)

예시 없이 "JSON으로"만 요청하면 모델이 구조를 마음대로 정합니다.

In [ ]:
zeroshot_prompt = [{"role": "user", "content": "Create a character profile for an RPG game in JSON format."}]
outputs = pipe(zeroshot_prompt)
print(outputs[0]["generated_text"])

원하는 키와 자리표시 값을 예시로 보여 주면(one-shot) 그 구조를 따릅니다. 다만 따를지 말지는 여전히 모델에게 달려 있습니다(책 273쪽).

In [ ]:
one_shot_template = """Create a short character profile for an RPG game. Make sure to only use this format:

{
  "description": "A SHORT DESCRIPTION",
  "name": "THE CHARACTER'S NAME",
  "armor": "ONE PIECE OF ARMOR",
  "weapon": "ONE OR MORE WEAPONS"
}
"""
one_shot_prompt = [{"role": "user", "content": one_shot_template}]
outputs = pipe(one_shot_prompt)
print(outputs[0]["generated_text"])

### 6-2. (+ 보충) Agent식 출력 검증: 파싱 → 검사 → 재시도

Agent가 LLM 출력을 받아 다음 행동(도구 호출, 제어 명령)을 정하려면 출력이 **기계가 읽을 수 있어야** 합니다. 그래서 실제 시스템은 (1) JSON 파싱, (2) 필수 키와 허용 값 검사, (3) 실패하면 오류 내용을 알려 주고 다시 요청하는 루프를 둡니다. 아래는 주행 상황을 받아 고수준 행동을 고르는 가상의 판단기입니다(실제 차량 제어용이 아닌 설명용 예).

In [ ]:
import json

ACTIONS = ["keep_lane", "slow_down", "stop", "change_lane_left", "change_lane_right"]
RISK = ["low", "medium", "high"]
instruction = (
    "You are the decision module of an autonomous vehicle. Read the scene and reply with ONLY a JSON object "
    f'with keys "risk" (one of {RISK}), "action" (one of {ACTIONS}) and "reason" (one short sentence). No other text.'
)

def validate(text):
    s, e = text.find("{"), text.rfind("}")
    if s < 0 or e < 0:
        return None, "no JSON object found"
    try:
        obj = json.loads(text[s:e + 1])
    except json.JSONDecodeError as err:
        return None, f"invalid JSON: {err}"
    missing = {"risk", "action", "reason"} - obj.keys()
    if missing:
        return None, f"missing keys: {sorted(missing)}"
    if obj["action"] not in ACTIONS or obj["risk"] not in RISK:
        return None, f"value not allowed: action={obj['action']!r}, risk={obj['risk']!r}"
    return obj, None

def decide(scene, max_tries=3):
    msgs = [{"role": "user", "content": f"{instruction}\n\nScene: {scene}"}]
    for attempt in range(1, max_tries + 1):
        out = pipe(msgs, max_new_tokens=120)[0]["generated_text"]
        obj, err = validate(out)
        print(f"  try {attempt}: {'OK' if obj else 'FAIL - ' + err}")
        if obj:
            return obj
        msgs += [{"role": "assistant", "content": out},
                 {"role": "user", "content": f"Your reply was rejected ({err}). Reply again with ONLY the JSON object."}]
    return {"risk": "high", "action": "stop", "reason": "fallback: model output could not be validated"}

for scene in ["A child is chasing a ball toward the road 20 m ahead; we drive at 40 km/h in a residential area.",
              "Empty highway, clear weather, lane markings visible, speed 100 km/h.",
              "A truck in the right lane is signaling and slowly merging into our lane."]:
    print(scene); print("  →", decide(scene))

관찰 포인트: 몇 번째 시도에서 통과하는지, 그리고 검증에 끝내 실패하면 **안전한 기본값(stop)**으로 떨어지게 만든 점을 보세요. 검증 없이 LLM 출력을 그대로 실행에 넘기지 않는 것이 Agent 설계의 기본입니다.

### 6-3. 문법 제약 샘플링: llama.cpp (PDF 274–277)

6-2는 출력을 **받은 뒤** 검사합니다. 문법 제약(constrained sampling)은 한 걸음 더 나아가, 토큰을 고르는 단계에서 문법에 맞지 않는 토큰을 **아예 후보에서 뺍니다**. 그래서 출력이 항상 유효한 JSON이 됩니다. 책은 llama.cpp의 JSON 문법을 씁니다. 같은 모델을 GGUF 형식(llama.cpp용 파일 형식)으로 다시 불러오므로, 먼저 transformers 모델을 GPU 메모리에서 내립니다.

In [ ]:
import gc
del model, tokenizer, pipe
gc.collect(); torch.cuda.empty_cache()

llama-cpp-python을 설치합니다. 책 공식 노트북은 소스 빌드(`CMAKE_ARGS="-DLLAMA_CUBLAS=on" pip install llama-cpp-python`)를 쓰는데, 이 옵션 이름은 이후 llama.cpp에서 `GGML_CUDA`로 바뀌었고 빌드에 시간이 오래 걸립니다. 여기서는 llama-cpp-python 프로젝트가 배포하는 **미리 빌드된 wheel**을 받습니다(GPU면 CUDA 12.4용, 아니면 CPU용). 버전은 2026-11 확인 시점의 0.3.36으로 고정했습니다.

> CUDA wheel을 설치했는데 `import llama_cpp`에서 CUDA 라이브러리 관련 오류가 나면, 이미지의 CUDA 런타임과 맞지 않는 경우입니다. 그때는 `idx = "cpu"`로 바꿔 다시 설치하고 아래 `n_gpu_layers=0`, `GGUF_FILE = "*q4.gguf"`로 실행하세요(느리지만 동작합니다).

In [ ]:
if importlib.util.find_spec("llama_cpp") is None:
    idx = "cu124" if torch.cuda.is_available() else "cpu"
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "llama-cpp-python==0.3.36",
                           "--only-binary=llama-cpp-python",
                           "--extra-index-url", f"https://abetlen.github.io/llama-cpp-python/whl/{idx}"])
import llama_cpp
print("llama-cpp-python", llama_cpp.__version__)

`n_gpu_layers=-1`은 모든 층을 GPU에 올린다는 뜻입니다. 책은 fp16 파일(7.6GB)을 씁니다. 디스크나 VRAM이 빠듯하면 4비트 양자화 파일(q4, 2.4GB)을 고르세요. 양자화는 7장과 12장에서 다룹니다.

In [ ]:
from llama_cpp.llama import Llama

GGUF_REPO = "microsoft/Phi-3-mini-4k-instruct-gguf"
GGUF_FILE = "*fp16.gguf"          # 책. 공간을 아끼려면 "*q4.gguf"
llm = Llama.from_pretrained(
    repo_id=GGUF_REPO,
    filename=GGUF_FILE,
    n_gpu_layers=-1 if torch.cuda.is_available() else 0,
    n_ctx=2048,
    verbose=False,
)

`response_format={"type": "json_object"}`을 주면 llama.cpp가 내부 JSON 문법을 적용합니다.

In [ ]:
output = llm.create_chat_completion(
    messages=[{"role": "user", "content": "Create a warrior for an RPG in JSON format."}],
    response_format={"type": "json_object"},
    temperature=0,
)['choices'][0]['message']["content"]

문법 덕분에 `json.loads`가 실패하지 않습니다. 단, 생성이 끝까지 진행될 때만입니다. `max_tokens`를 작게 주어 중간에 끊기면 문법이 보장하던 JSON도 닫히지 않은 채 끝나 `JSONDecodeError`가 납니다(제 작업 환경에서 `max_tokens=120`으로 시험해 확인).

In [ ]:
import json
json_output = json.dumps(json.loads(output), indent=4)
print(json_output)

### 6-4. (+ 보충) JSON 스키마로 키와 허용 값까지 제약

`json_object`는 "유효한 JSON"만 보장하고, 어떤 키가 나올지는 보장하지 않습니다. llama-cpp-python은 `response_format`에 JSON 스키마를 함께 넘기면 그 스키마를 문법으로 바꿔 적용합니다. 6-2의 판단기를 스키마 제약으로 다시 만들면, 재시도 루프 없이도 `action`이 허용 목록 밖의 값이 될 수 없습니다.

In [ ]:
schema = {
    "type": "object",
    "properties": {
        "risk": {"type": "string", "enum": RISK},
        "action": {"type": "string", "enum": ACTIONS},
        "reason": {"type": "string"},
    },
    "required": ["risk", "action", "reason"],
}
scene = "A child is chasing a ball toward the road 20 m ahead; we drive at 40 km/h in a residential area."
out = llm.create_chat_completion(
    messages=[{"role": "user", "content": f"{instruction}\n\nScene: {scene}"}],
    response_format={"type": "json_object", "schema": schema},
    temperature=0, max_tokens=120,
)["choices"][0]["message"]["content"]
obj = json.loads(out)
print(obj, "| 검증:", validate(out)[1] or "OK")

**한계.** 문법은 **형식**만 보장합니다. "아이가 뛰어드는데 keep_lane"처럼 형식은 맞고 판단은 틀린 출력은 막지 못합니다. 실제로 제 작업 환경에서 작은 모델(Qwen2.5 0.5B, q4)로 이 셀을 시험했을 때 스키마 검증은 통과했지만 `action`은 `keep_lane`이었습니다. 내용의 정확성은 책이 말하는 정확성(accuracy) 검증, 즉 별도 평가나 규칙 기반 안전장치가 맡아야 합니다.

## 7. 정리

| 기법 | 하는 일 | 이 노트북 |
|---|---|---|
| temperature / top_p | 샘플링의 무작위성과 후보 폭 조절 | 2 |
| 프롬프트 재료 조립 | 페르소나·지시·맥락·형식·청중·어조·데이터 | 4-1 |
| 인컨텍스트 학습 | 예시로 원하는 행동을 보여 줌 | 4-2 |
| 체인 프롬프팅 | 문제를 단계로 나눠 출력을 이어 붙임 | 4-3 |
| CoT / Zero-shot CoT / ToT | 답 전에 풀이를 생성하게 함 | 5 |
| Self-Consistency | 여러 번 샘플링해 다수결 | 5-4 |
| 출력 검증 | 예시, 파싱·검사·재시도, 문법·스키마 제약 | 6 |

**자율주행·Agent로 이어지는 점(책 밖).** 이 장의 기법은 Agent의 기본 부품입니다. 체인 프롬프팅은 7장의 Chain, CoT와 6-2의 검증·재시도 루프는 7장 ReAct Agent의 "생각 → 행동 → 관찰" 반복으로 이어집니다. 6-3·6-4의 제약 디코딩은 LLM이 도구 호출(tool call)이나 제어 명령처럼 **형식이 틀리면 안 되는 출력**을 낼 때 쓰는 방식입니다. 주행 판단에 LLM·VLM을 쓰는 연구(예: 장면 설명 → 고수준 행동 선택)에서도 출력 형식 강제와 규칙 기반 안전 검증을 함께 두는 구성이 일반적입니다. 5절의 CoT는 2024년 이후 등장한 추론 모델(reasoning model)이 학습 단계에서 아예 내재화한 행동이기도 합니다.

**끝나면** Studio에서 space를 **Stop** 하세요. GGUF 파일(최대 7.6GB)이 EBS에 남으니, 공간이 부족하면 `~/.cache/huggingface/hub`에서 해당 폴더를 지우세요.